In [27]:
   import pandas as pd
   pd.__version__

'3.0.6'

In [28]:
import numpy as np
import pandas as pd

RAW = "data_raw/"
courses = pd.read_csv(RAW + "courses.csv", na_values=["?"])
assess  = pd.read_csv(RAW + "assessments.csv", na_values=["?"])
info    = pd.read_csv(RAW + "studentInfo.csv", na_values=["?"])
reg     = pd.read_csv(RAW + "studentRegistration.csv", na_values=["?"])
subs    = pd.read_csv(RAW + "studentAssessment.csv", na_values=["?"])
vle     = pd.read_csv(RAW + "studentVle.csv", na_values=["?"],
                      usecols=["code_module", "code_presentation", "id_student", "date", "sum_click"])

for name, df in [("courses", courses), ("assessments", assess), ("studentInfo", info),
                 ("studentRegistration", reg), ("studentAssessment", subs), ("studentVle", vle)]:
    print(f"{name:20} {df.shape[0]:>12,} rows   {df.shape[1]} columns")

courses                        22 rows   3 columns
assessments                   206 rows   6 columns
studentInfo                32,593 rows   12 columns
studentRegistration        32,593 rows   5 columns
studentAssessment         173,912 rows   5 columns
studentVle             10,655,280 rows   5 columns


In [29]:
info.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


In [30]:
info["final_result"].value_counts()

final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

In [31]:
KEY = ["code_module", "code_presentation", "id_student"]   # these 3 columns identify one enrolment

print("Duplicate enrolments:", info.duplicated(KEY).sum())

check = info.merge(reg, on=KEY, how="outer", indicator=True)
check["_merge"].value_counts()

Duplicate enrolments: 0


_merge
both          32593
left_only         0
right_only        0
Name: count, dtype: int64

In [32]:
for name, df in [("studentInfo", info), ("studentRegistration", reg),
                 ("studentAssessment", subs), ("assessments", assess)]:
    missing = df.isna().sum()
    print(f"{name:20}", missing[missing > 0].to_dict())

studentInfo          {'imd_band': 1111}
studentRegistration  {'date_registration': 45, 'date_unregistration': 22521}
studentAssessment    {'score': 173}
assessments          {'date': 11}


In [33]:
info["imd_band"].value_counts(dropna=False).sort_index()

imd_band
0-10%      3311
10-20      3516
20-30%     3654
30-40%     3539
40-50%     3256
50-60%     3124
60-70%     2905
70-80%     2879
80-90%     2762
90-100%    2536
NaN        1111
Name: count, dtype: int64

In [34]:
enr = info.merge(reg, on=KEY)
pd.crosstab(enr["final_result"], enr["date_unregistration"].notna())

date_unregistration,False,True
final_result,,
Distinction,3024,0
Fail,7043,9
Pass,12361,0
Withdrawn,93,10063


In [35]:
courses = courses.sort_values(["code_module", "code_presentation"]).reset_index(drop=True)
courses["CourseKey"] = courses.index + 1

DOMAIN = {"AAA": "Social Sciences", "BBB": "Social Sciences", "GGG": "Social Sciences",
          "CCC": "STEM", "DDD": "STEM", "EEE": "STEM", "FFF": "STEM"}

dim_course = pd.DataFrame({
    "CourseKey":        courses["CourseKey"],
    "CourseCode":       courses["code_module"] + "-" + courses["code_presentation"],
    "ModuleCode":       courses["code_module"],
    "PresentationCode": courses["code_presentation"],
    "Domain":           courses["code_module"].map(DOMAIN),
    "LengthDays":       courses["module_presentation_length"].astype(int),
})
dim_course

,CourseKey,CourseCode,ModuleCode,PresentationCode,Domain,LengthDays
0,1,AAA-2013J,AAA,2013J,Social Sciences,268
1,2,AAA-2014J,AAA,2014J,Social Sciences,269
2,3,BBB-2013B,BBB,2013B,Social Sciences,240
3,4,BBB-2013J,BBB,2013J,Social Sciences,268
4,5,BBB-2014B,BBB,2014B,Social Sciences,234
5,6,BBB-2014J,BBB,2014J,Social Sciences,262
6,7,CCC-2014B,CCC,2014B,STEM,241
7,8,CCC-2014J,CCC,2014J,STEM,269
8,9,DDD-2013B,DDD,2013B,STEM,240
9,10,DDD-2013J,DDD,2013J,STEM,261


In [36]:
dim_result = pd.DataFrame({
    "ResultKey":   [1, 2, 3, 4],
    "FinalResult": ["Distinction", "Pass", "Fail", "Withdrawn"],
    "ResultGroup": ["Passed", "Passed", "Failed", "Withdrawn"],
})
RESULT_KEY = dict(zip(dim_result["FinalResult"], dim_result["ResultKey"]))
RESULT_KEY

{'Distinction': 1, 'Pass': 2, 'Fail': 3, 'Withdrawn': 4}

In [37]:
for col in ["gender", "region", "highest_education", "imd_band", "disability", "age_band"]:
    n = (info.groupby("id_student")[col].nunique(dropna=False) > 1).sum()
    print(f"{col:18} students with more than one value: {n}")

gender             students with more than one value: 0
region             students with more than one value: 0
highest_education  students with more than one value: 0
imd_band           students with more than one value: 0
disability         students with more than one value: 0
age_band           students with more than one value: 72


In [38]:
stu = info.drop_duplicates("id_student")

EDU_ORDER = {"No Formal quals": 1, "Lower Than A Level": 2, "A Level or Equivalent": 3,
             "HE Qualification": 4, "Post Graduate Qualification": 5}

dim_student = pd.DataFrame({
    "StudentID":        stu["id_student"],
    "Gender":           stu["gender"].map({"M": "Male", "F": "Female"}),
    "Region":           stu["region"],
    "HighestEducation": stu["highest_education"],
    "EducationOrder":   stu["highest_education"].map(EDU_ORDER),
    "IMDBand":          stu["imd_band"].replace({"10-20": "10-20%"}).fillna("Unknown"),
    "Disability":       stu["disability"].map({"Y": "Yes", "N": "No"}),
})
print("Students:", len(dim_student))
dim_student["IMDBand"].value_counts().sort_index()

Students: 28785


IMDBand
0-10%      2944
10-20%     3132
20-30%     3200
30-40%     3128
40-50%     2890
50-60%     2787
60-70%     2560
70-80%     2532
80-90%     2421
90-100%    2220
Unknown     971
Name: count, dtype: int64

In [39]:
CRS = ["code_module", "code_presentation"]   # identifies one course run

enr = (info.merge(reg, on=KEY)
           .merge(courses[CRS + ["CourseKey"]], on=CRS)
           .sort_values(["CourseKey", "id_student"])
           .reset_index(drop=True))

withdrawn = enr["final_result"] == "Withdrawn"
has_date  = enr["date_unregistration"].notna()

week = np.floor(enr["date_unregistration"] / 7)
print("Withdrew 4+ weeks before start:", (withdrawn & has_date & (week < -4)).sum())
print("Withdrew after week 38:        ", (withdrawn & has_date & (week > 38)).sum())

enr["WithdrawalWeek"] = week.clip(-4, 38).where(withdrawn & has_date)
enr["WithdrawalWeek"].describe()

Withdrew 4+ weeks before start: 1205
Withdrew after week 38:         1


count    10063.000000
mean         7.441320
std         10.463864
min         -4.000000
25%         -1.000000
50%          3.000000
75%         15.000000
max         38.000000
Name: WithdrawalWeek, dtype: float64

In [40]:
first = (assess[assess["assessment_type"] != "Exam"]
         .sort_values(["date", "id_assessment"])
         .groupby(CRS).head(1)[CRS + ["id_assessment", "date"]]
         .rename(columns={"id_assessment": "FirstAssessmentID"}))
first["CheckpointDay"] = (first["date"] + 7).astype(int)
first = first.drop(columns="date").sort_values(CRS)
first

,code_module,code_presentation,FirstAssessmentID,CheckpointDay
0,AAA,2013J,1752,26
6,AAA,2014J,1758,26
17,BBB,2013B,14984,26
29,BBB,2013J,14996,26
41,BBB,2014B,15008,19
48,BBB,2014J,15020,26
54,CCC,2014B,24286,25
64,CCC,2014J,24295,25
74,DDD,2013B,25341,30
88,DDD,2013J,25348,32


In [41]:
enr = enr.merge(first, on=CRS, how="left")
left_before = withdrawn & has_date & (enr["date_unregistration"] < enr["CheckpointDay"])
enr["ActiveAtCheckpoint"] = np.where(left_before, "No", "Yes")

v = vle.merge(first[CRS + ["CheckpointDay"]], on=CRS)
early = v[v["date"] < v["CheckpointDay"]].groupby(KEY)["sum_click"].sum().rename("EarlyClicks")
enr = enr.merge(early, left_on=KEY, right_index=True, how="left")
enr["EarlyClicks"] = enr["EarlyClicks"].fillna(0).astype(int)

active = enr["ActiveAtCheckpoint"] == "Yes"
pct = enr[active].groupby("CourseKey")["EarlyClicks"].rank(pct=True)
enr["EarlyEngagement"] = "Left before checkpoint"
enr.loc[active, "EarlyEngagement"] = np.select([pct <= 0.25, pct > 0.75], ["Low", "High"], "Medium")
enr["EarlyEngagement"].value_counts()

EarlyEngagement
Medium                    13837
High                       6916
Low                        6887
Left before checkpoint     4953
Name: count, dtype: int64

In [42]:
fa = subs.merge(first, left_on="id_assessment", right_on="FirstAssessmentID")
fa = fa[fa["date_submitted"] <= fa["CheckpointDay"]][KEY + ["score"]]
fa["FirstAssessmentStatus"] = np.where(fa["score"] < 40, "Below 40", "40+")
enr = enr.merge(fa, on=KEY, how="left")
enr["FirstAssessmentStatus"] = enr["FirstAssessmentStatus"].fillna("Not submitted")
enr["FirstAssessmentStatus"].value_counts()

FirstAssessmentStatus
40+              23202
Not submitted     7877
Below 40          1514
Name: count, dtype: int64

In [43]:
fact_enrollment = pd.DataFrame({
    "EnrollmentID":          np.arange(1, len(enr) + 1),
    "CourseKey":             enr["CourseKey"],
    "StudentID":             enr["id_student"],
    "ResultKey":             enr["final_result"].map(RESULT_KEY),
    "AgeBand":               enr["age_band"].replace({"55<=": "55+"}),
    "NumPrevAttempts":       enr["num_of_prev_attempts"],
    "StudiedCredits":        enr["studied_credits"],
    "RegistrationDay":       enr["date_registration"].astype("Int64"),
    "WithdrawalDay":         enr["date_unregistration"].where(withdrawn).astype("Int64"),
    "WithdrawalWeek":        enr["WithdrawalWeek"].astype("Int64"),
    "CheckpointDay":         enr["CheckpointDay"],
    "ActiveAtCheckpoint":    enr["ActiveAtCheckpoint"],
    "EarlyClicks":           enr["EarlyClicks"],
    "EarlyEngagement":       enr["EarlyEngagement"],
    "FirstAssessmentScore":  enr["score"].astype("Int64"),
    "FirstAssessmentStatus": enr["FirstAssessmentStatus"],
})
print(fact_enrollment.shape)
fact_enrollment.head()

(32593, 16)


,EnrollmentID,CourseKey,StudentID,ResultKey,AgeBand,NumPrevAttempts,StudiedCredits,RegistrationDay,WithdrawalDay,WithdrawalWeek,CheckpointDay,ActiveAtCheckpoint,EarlyClicks,EarlyEngagement,FirstAssessmentScore,FirstAssessmentStatus
0,1,1,11391,2,55+,0,240,-159,<NA>,<NA>,26,Yes,401,Medium,78,40+
1,2,1,28400,2,35-55,0,60,-53,<NA>,<NA>,26,Yes,550,Medium,70,40+
2,3,1,30268,4,35-55,0,60,-92,12,1,26,No,281,Left before checkpoint,<NA>,Not submitted
3,4,1,31604,2,35-55,0,60,-52,<NA>,<NA>,26,Yes,478,Medium,72,40+
4,5,1,32885,2,0-35,0,60,-176,<NA>,<NA>,26,Yes,558,Medium,69,40+


In [44]:
pd.crosstab(enr["EarlyEngagement"], enr["final_result"], normalize="index").round(2)

final_result,Distinction,Fail,Pass,Withdrawn
EarlyEngagement,,,,
High,0.21,0.13,0.54,0.12
Left before checkpoint,0.00,0.00,0.00,1.00
Low,0.03,0.42,0.28,0.27
Medium,0.10,0.23,0.48,0.18


In [45]:
a = assess.merge(courses[CRS + ["CourseKey", "module_presentation_length"]], on=CRS)
print("Exams with no date:", a["date"].isna().sum())

a["DueDay"] = a["date"].fillna(a["module_presentation_length"]).astype(int)
a = a.sort_values(["CourseKey", "assessment_type", "DueDay", "id_assessment"])
a["Seq"] = a.groupby(["CourseKey", "assessment_type"]).cumcount() + 1

dim_assessment = pd.DataFrame({
    "AssessmentID":    a["id_assessment"],
    "AssessmentLabel": a["assessment_type"] + " " + a["Seq"].astype(str),
    "AssessmentType":  a["assessment_type"].map({"TMA": "Tutor-marked", "CMA": "Computer-marked", "Exam": "Final exam"}),
    "DueDay":          a["DueDay"],
    "DueWeek":         a["DueDay"] // 7,
    "Weight":          a["weight"],
})
print(dim_assessment.shape)
dim_assessment.head()

Exams with no date: 11
(206, 6)


,AssessmentID,AssessmentLabel,AssessmentType,DueDay,DueWeek,Weight
5,1757,Exam 1,Final exam,268,38,100.0
0,1752,TMA 1,Tutor-marked,19,2,10.0
1,1753,TMA 2,Tutor-marked,54,7,20.0
2,1754,TMA 3,Tutor-marked,117,16,20.0
3,1755,TMA 4,Tutor-marked,166,23,20.0


In [46]:
s = (subs.merge(a[["id_assessment", "code_module", "code_presentation", "CourseKey", "assessment_type", "DueDay"]],
                on="id_assessment")
         .merge(enr[KEY + ["final_result"]], on=KEY, how="left"))
print("Submissions without a matching enrolment:", s["final_result"].isna().sum())

late_applies = (s["assessment_type"] == "TMA") & (s["is_banked"] == 0)
fact_assessment = pd.DataFrame({
    "AssessmentID": s["id_assessment"],
    "CourseKey":    s["CourseKey"],
    "StudentID":    s["id_student"],
    "ResultKey":    s["final_result"].map(RESULT_KEY),
    "SubmittedDay": s["date_submitted"],
    "DaysLate":     (s["date_submitted"] - s["DueDay"]).where(late_applies).astype("Int64"),
    "IsBanked":     s["is_banked"],
    "Score":        s["score"].astype("Int64"),
})
print(fact_assessment.shape)
late = fact_assessment["DaysLate"]
print(f"Late tutor-marked submissions: {(late > 0).sum() / late.notna().sum():.1%}")

Submissions without a matching enrolment: 0
(173912, 8)
Late tutor-marked submissions: 16.8%


In [47]:
vle["WeekNo"] = np.floor_divide(vle["date"], 7)
wk = (vle.groupby(KEY + ["WeekNo"])
         .agg(Clicks=("sum_click", "sum"), ActiveDays=("date", "nunique"))
         .reset_index()
         .merge(courses[CRS + ["CourseKey"]], on=CRS)
         .merge(enr[KEY + ["final_result"]], on=KEY, how="left"))

fact_vle_weekly = pd.DataFrame({
    "CourseKey":  wk["CourseKey"],
    "StudentID":  wk["id_student"],
    "ResultKey":  wk["final_result"].map(RESULT_KEY),
    "WeekNo":     wk["WeekNo"],
    "Clicks":     wk["Clicks"],
    "ActiveDays": wk["ActiveDays"],
})
print(fact_vle_weekly.shape)
print(f"Clicks in raw data:    {vle['sum_click'].sum():,}")
print(f"Clicks after grouping: {fact_vle_weekly['Clicks'].sum():,}")

(627031, 6)
Clicks in raw data:    39,605,099
Clicks after grouping: 39,605,099


In [48]:
w = np.arange(fact_vle_weekly["WeekNo"].min(), fact_vle_weekly["WeekNo"].max() + 1)
rules = [w < 0, w <= 4, w <= 12, w <= 25]
dim_week = pd.DataFrame({
    "WeekNo":     w,
    "Phase":      np.select(rules, ["Before start", "Weeks 0-4", "Weeks 5-12", "Weeks 13-25"], "Weeks 26-38"),
    "PhaseOrder": np.select(rules, [1, 2, 3, 4], 5),
})
print(len(dim_week), "weeks, from", w.min(), "to", w.max())
dim_week.groupby(["PhaseOrder", "Phase"]).size()

43 weeks, from -4 to 38


PhaseOrder  Phase       
1           Before start     4
2           Weeks 0-4        5
3           Weeks 5-12       8
4           Weeks 13-25     13
5           Weeks 26-38     13
dtype: int64

In [49]:
checks = [
    ("Enrolments",  len(info),                       len(fact_enrollment)),
    ("Students",    info["id_student"].nunique(),    len(dim_student)),
    ("Submissions", len(subs),                       len(fact_assessment)),
    ("Clicks",      vle["sum_click"].sum(),          fact_vle_weekly["Clicks"].sum()),
    ("Withdrawn",   (info["final_result"] == "Withdrawn").sum(), (fact_enrollment["ResultKey"] == 4).sum()),
]
for label, source, output in checks:
    status = "OK" if source == output else "MISMATCH"
    print(f"{label:12} source {source:>12,}   output {output:>12,}   {status}")

for name, fact in [("fact_enrollment", fact_enrollment), ("fact_assessment", fact_assessment),
                   ("fact_vle_weekly", fact_vle_weekly)]:
    orphans = ((~fact["CourseKey"].isin(dim_course["CourseKey"])).sum()
               + (~fact["StudentID"].isin(dim_student["StudentID"])).sum())
    print(f"Orphan keys in {name}: {orphans}")

Enrolments   source       32,593   output       32,593   OK
Students     source       28,785   output       28,785   OK
Submissions  source      173,912   output      173,912   OK
Clicks       source   39,605,099   output   39,605,099   OK
Withdrawn    source       10,156   output       10,156   OK
Orphan keys in fact_enrollment: 0
Orphan keys in fact_assessment: 0
Orphan keys in fact_vle_weekly: 0


In [50]:
import os
OUT = "my_data_clean/"
os.makedirs(OUT, exist_ok=True)

tables = {"dim_course": dim_course, "dim_student": dim_student, "dim_result": dim_result,
          "dim_assessment": dim_assessment, "dim_week": dim_week,
          "fact_enrollment": fact_enrollment, "fact_assessment": fact_assessment,
          "fact_vle_weekly": fact_vle_weekly}

for name, table in tables.items():
    table.to_csv(OUT + name + ".csv", index=False)
    print(f"Saved {name + '.csv':22} {len(table):>9,} rows")

Saved dim_course.csv                22 rows
Saved dim_student.csv           28,785 rows
Saved dim_result.csv                 4 rows
Saved dim_assessment.csv           206 rows
Saved dim_week.csv                  43 rows
Saved fact_enrollment.csv       32,593 rows
Saved fact_assessment.csv      173,912 rows
Saved fact_vle_weekly.csv      627,031 rows


In [51]:
for name in tables:
    yours = pd.read_csv("my_data_clean/" + name + ".csv")
    mine  = pd.read_csv("data_clean/" + name + ".csv")
    print(f"{name:16} {'identical' if yours.equals(mine) else 'DIFFERENT'}")

dim_course       identical
dim_student      identical
dim_result       identical
dim_assessment   identical
dim_week         identical
fact_enrollment  identical
fact_assessment  identical
fact_vle_weekly  identical
